## Decoder-Only Transformer for Text Generation in PyTorch

* In this notebook, we implement and train a decoder-only Transformer model using PyTorch for character-level sequence modeling. Starting with a set of 25 sample sentences, we build a vocabulary and encode the text into numerical representations. The data is padded and split into input and target sequences suitable for next-token prediction. We define a custom Transformer block with masked self-attention to ensure the model only attends to previous tokens, mimicking autoregressive text generation. The DecoderModel stacks multiple attention blocks and outputs logits for each character in the vocabulary. After initializing the model, optimizer, and loss function, we train the network over several epochs, monitoring the loss. Finally, we demonstrate text generation by sampling new sequences from the trained model, starting from a given prompt. This notebook provides a hands-on example of building, training, and evaluating a Transformer for sequence generation tasks, highlighting key concepts such as attention masking, batching, and autoregressive decoding.


**Key Concepts:**
* Model Type: Transformer-based Decoder (advanced)
* Context Used: All previous characters in sequence (autoregressive)
* Architecture: Multi-layer masked self-attention (Transformer blocks)
* Dependencies Modeled: Long-range dependencies using attention
* Purpose: Demonstrate full Transformer for sequence generation
* Generation Style: Autoregressive, learned attention over previous tokens

In [1]:
# PyTorch Decoder-Only Transformer Training Example
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
# Sample data: 25 larger sentences (character-level)
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "PyTorch makes building neural networks easy and flexible.",
    "Transformers are powerful models for sequence tasks.",
    "Self-attention enables models to focus on relevant parts.",
    "Sequence modeling is essential for natural language processing.",
    "Decoder-only architectures generate text one token at a time.",
    "Training deep learning models requires lots of data and compute.",
    "Model evaluation helps track progress and avoid overfitting.",
    "Sample sentences provide a simple way to test algorithms.",
    "Data preprocessing is a key step in any ML pipeline.",
    "Records in datasets should be clean and well formatted.",
    "Blocks of code are easier to debug when modularized.",
    "Self-supervised learning is common in NLP tasks.",
    "Feature engineering can improve model performance.",
    "Scratch implementations help understand core concepts.",
    "Tokenization splits text into manageable pieces for models.",
    "Embedding layers convert tokens into dense vectors.",
    "Output layers map hidden states to vocabulary predictions.",
    "Input data must be batched for efficient GPU training.",
    "Layer normalization stabilizes training in deep networks.",
    "Batch size affects convergence and memory usage.",
    "Loss functions guide the optimization process.",
    "Optimizers like Adam are popular for training neural nets.",
    "Forward and backward passes compute gradients and update weights.",
    "Backpropagation is the foundation of neural network learning."
]

In [3]:

# Build vocabulary
all_text = "".join(sentences)
chars = sorted(list(set(all_text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

In [4]:
encode("Forward")

[7, 33, 36, 41, 19, 36, 22]

In [5]:
# Prepare data (pad to max length)
max_len = max(len(s) for s in sentences)
inputs = []
targets = []
for s in sentences:
    x = encode(s.ljust(max_len))[:-1]  # input (all but last)
    y = encode(s.ljust(max_len))[1:]   # target (all but first)
    inputs.append(x)
    targets.append(y)
inputs = torch.tensor(inputs, dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)


In [59]:
max_len

65

In [6]:
inputs[0]

tensor([17, 26, 23,  0, 35, 39, 27, 21, 29,  0, 20, 36, 33, 41, 32,  0, 24, 33,
        42,  0, 28, 39, 31, 34, 37,  0, 33, 40, 23, 36,  0, 38, 26, 23,  0, 30,
        19, 44, 43,  0, 22, 33, 25,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [7]:
targets[0]

tensor([26, 23,  0, 35, 39, 27, 21, 29,  0, 20, 36, 33, 41, 32,  0, 24, 33, 42,
         0, 28, 39, 31, 34, 37,  0, 33, 40, 23, 36,  0, 38, 26, 23,  0, 30, 19,
        44, 43,  0, 22, 33, 25,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [8]:
# Decoder-only Transformer block
class SelfAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.ReLU(),
            nn.Linear(embed_dim*4, embed_dim)
        )
        self.ln2 = nn.LayerNorm(embed_dim)
    def forward(self, x):
        self.attn_mask = torch.triu(torch.ones(x.size(1), x.size(1)), diagonal=1).bool().to(x.device)
        self.attn_out, _ = self.attn(x, x, x, attn_mask=self.attn_mask)
        self.x = self.ln1(x + self.attn_out)
        self.ff_out = self.ff(self.x)
        self.x = self.ln2(x + self.ff_out)
        return self.x

class DecoderModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=2, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.blocks = nn.ModuleList([
            SelfAttentionBlock(embed_dim, num_heads) for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(embed_dim, vocab_size)
    def forward(self, x):
        x = self.embed(x)
        for block in self.blocks:
            x = block(x)
        logits = self.fc_out(x)
        return logits



In [44]:
model = DecoderModel(vocab_size, embed_dim=128, num_heads=8, num_layers=4)
model

DecoderModel(
  (embed): Embedding(45, 128)
  (blocks): ModuleList(
    (0-3): 4 x SelfAttentionBlock(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=128, bias=True)
      )
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (fc_out): Linear(in_features=128, out_features=45, bias=True)
)

In [28]:
inputs[0].unsqueeze(0).shape, inputs.shape

(torch.Size([1, 64]), torch.Size([25, 64]))

In [29]:
_ = model(inputs[0].unsqueeze(0))  # Run a forward pass with batch dimension
attn_mask = model.blocks[0].attn_mask
attn_mask.shape

torch.Size([64, 64])

In [31]:
attn_mask

tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])

In [32]:
inputs[0]
# First row of attention mask says that model seen only token number 17 and will predict 26th token from targets[0]
# now, in this case, the attention is calculated only for the first token, so it can only see itself and not the next token.
# This is because the model is decoder-only and the attention mask prevents it from looking ahead.

## Then second rowof attention mask says 2 False, i.e it will see token 17th and token 26th and will predict tken 23 from targets[0]
# Now, the second token can see itself and the first token, but not the third token.

# So, we  do not need to create separate examples for each prefix.
# The model learns to use all previous tokens as context for each prediction

tensor([17, 26, 23,  0, 35, 39, 27, 21, 29,  0, 20, 36, 33, 41, 32,  0, 24, 33,
        42,  0, 28, 39, 31, 34, 37,  0, 33, 40, 23, 36,  0, 38, 26, 23,  0, 30,
        19, 44, 43,  0, 22, 33, 25,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [33]:
targets[0]

tensor([26, 23,  0, 35, 39, 27, 21, 29,  0, 20, 36, 33, 41, 32,  0, 24, 33, 42,
         0, 28, 39, 31, 34, 37,  0, 33, 40, 23, 36,  0, 38, 26, 23,  0, 30, 19,
        44, 43,  0, 22, 33, 25,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

In [34]:
model.blocks[0].attn_out.shape

torch.Size([1, 64, 128])

In [35]:
# In this context:

# - **1** is the batch size, which corresponds to the number of sentences in your dataset (`len(sentences)`).
# - **64** is the sequence length, which is `max_len - 1` (since you pad sentences to `max_len` and then remove one for input/target alignment).
# - **128** is the embedding dimension (`embed_dim`) used in your transformer model.

# So, a tensor with shape `(25, 64, 32)` would represent a batch of 25 sequences, each of length 64, with each token represented by a 32-dimensional embedding.

In [36]:
inputs[0], targets[0]  # Example input and target

(tensor([17, 26, 23,  0, 35, 39, 27, 21, 29,  0, 20, 36, 33, 41, 32,  0, 24, 33,
         42,  0, 28, 39, 31, 34, 37,  0, 33, 40, 23, 36,  0, 38, 26, 23,  0, 30,
         19, 44, 43,  0, 22, 33, 25,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0]),
 tensor([26, 23,  0, 35, 39, 27, 21, 29,  0, 20, 36, 33, 41, 32,  0, 24, 33, 42,
          0, 28, 39, 31, 34, 37,  0, 33, 40, 23, 36,  0, 38, 26, 23,  0, 30, 19,
         44, 43,  0, 22, 33, 25,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0]))

In [45]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [46]:
num_params = sum(p.numel() for p in model.parameters())
print("Number of parameters in model:", num_params)

Number of parameters in model: 804653


In [47]:
%%time 
# Training loop example

num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(inputs)
    loss = loss_fn(logits.view(-1, vocab_size), targets.view(-1))
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 4.0687
Epoch 20, Loss: 2.2321
Epoch 40, Loss: 1.8608
Epoch 60, Loss: 1.3379
Epoch 80, Loss: 0.7887
Epoch 100, Loss: 0.3484
Epoch 120, Loss: 0.1913
Epoch 140, Loss: 0.0652
Epoch 160, Loss: 0.0355
Epoch 180, Loss: 0.0258
CPU times: total: 2min 38s
Wall time: 43.5 s


In [53]:
def generate(model, start, max_new_tokens=40):
    model.eval()
    # idx = torch.tensor([encode(start.ljust(max_len))[:-1]], dtype=torch.long)
    idx = torch.tensor([encode(start)], dtype=torch.long)
    for _ in range(max_new_tokens):
        logits = model(idx)
        next_token = torch.argmax(logits[0, -1], dim=-1).item()
        idx = torch.cat([idx, torch.tensor([[next_token]])], dim=1)
    return decode(idx[0].tolist())



In [58]:
%%time
# Example generation
print("Generated:", generate(model, "Backpropagation is", 64))

Generated: Backpropagation is the foundation of neural network learning.                     
CPU times: total: 2.56 s
Wall time: 945 ms
